In [ ]:
from mssm.models import *
from mssmViz.sim import *
from mssmViz.plot import *
from mssmViz.extract import eval_coverage
import pickle
import copy
import os
import dotenv
dotenv.load_dotenv()

n_cores = int(os.getenv("n_cores"))

from defaults import (
    default_gamm_kwargs,
    default_gammlss_kwargs,
    default_gsmm_kwargs,
)

default_gamm_kwargs["n_cores"] = n_cores
default_gammlss_kwargs["n_cores"] = n_cores
default_gsmm_kwargs["n_cores"] = n_cores

qefs_kwargs = copy.deepcopy(default_gsmm_kwargs)
qefs_kwargs["method"] = "qEFS"

n_sim = 1
n_ranef = 25
fcoef_ratios = [0, 0.25, 0.5, 0.75]

# Sim 11 (sim 18 for Multinomial) (sim21 for Multigauss)

In [ ]:
############################# Simulation 3: Exponential #############################
sim_fams = [Gaussian(),Gamma(),Binomial(),Poisson()]
fam_names = ["Gaussian", "Gamma", "Binom", "Poisson"]
offsets = [0,0,-5,-10]

for should_correlate in [True,False]:

    for fam_name, sim_fam, b_offset in zip(fam_names,sim_fams,offsets):
        
        # Set up storage for current sim
        eta_mses = np.zeros((n_sim,1 + len(fcoef_ratios)))
        coverage = np.zeros((n_sim,1 + len(fcoef_ratios)))
        Failures = np.zeros((n_sim,1 + len(fcoef_ratios)))

        gsmm_fam = GAMLSSGSMMFamily(1,sim_fam)

        fcoef = 37 + (1 if sim_fam.twopar else 0)
        
        iterator = tqdm(range(n_sim),desc="Simulating",leave=True)
        for sim_i in iterator:

            sim_dat = sim11(5000,2,c=0,seed=sim_i,family=sim_fam,
                           binom_offset = b_offset,
                           correlate=should_correlate,
                           n_ranef=n_ranef)
            
            sim_dat.to_csv((f"./results/data/sim3_exp/sim_size:{n_sim}_fam:"
                            f"{fam_name}_corr:{should_correlate}_set:{sim_i}.csv"),index=False)

            # We need to model the mean: \mu_i - this time including a random smooth of x0 per level
            # of x4
            sim_formula_m = Formula(lhs("y"),
                                [i(),f(["x0"]),f(["x1"]),f(["x2"]),f(["x3"]),fs(["x0"],rf="x4")],
                                data=sim_dat)
            
            sim_i_failed = [False, *[False for _ in fcoef_ratios]]

            ############################# Fit model with EFS #############################
            model_efs = GAMM(copy.deepcopy(sim_formula_m),sim_fam)
            try:
                model_efs.fit(**default_gamm_kwargs)
            except:
                sim_i_failed[0] = True

            ############################# Fit model with qEFS #############################
            models = [model_efs]
            for rai,ratio in enumerate(fcoef_ratios):

                qefs_kwargs["structured_qefs_budget"] = int(ratio*fcoef)

                gsmm_model = GSMM(formulas=[copy.deepcopy(sim_formula_m)],family=gsmm_fam)
                
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")
                        
                        gsmm_model.fit(**qefs_kwargs)
                except:
                    sim_i_failed[1+rai] = True

                models.append(gsmm_model)
            
            ######################################## Collect MSEs ####################################

            for mi, model in enumerate(models):

                if sim_i_failed[mi]:
                    print(f"Model {mi+1} failed at {sim_i}")
                    Failures[sim_i,mi] = 1
                    eta_mses[sim_i,mi] = np.nan
                    coverage[sim_i,mi] = np.nan
                    continue

                # Not converged but not failed outright
                if model.info.code > 0:
                    Failures[sim_i,mi] = 1
                
                #plot(model)

                pred_diff = model.preds[0].flatten() - sim_dat["eta"].values
                _,mcov,_ = eval_coverage(model,sim_dat,target=sim_dat["eta"].values)
                coverage[sim_i,mi] = mcov
                eta_mses[sim_i,mi] = np.dot(pred_diff,pred_diff)/len(pred_diff)
            
            iterator.set_description_str(desc=f"MSE.: {[(float(np.round(m,decimals=4)),
                                                         float(np.round(sd,decimals=2)))
                                                         for m,sd in zip(np.mean(eta_mses[:(sim_i+1),:],axis=0),
                                                                         np.std(eta_mses[:(sim_i+1),:],axis=0))]}",
                                                                         refresh=True)
            ###################################### Save in progress results ######################################
            res = {"eta_mses":eta_mses,
                   "Failures":Failures,
                   "coverage":coverage,
                   }
            
            with open(f'./results/sim/sim3_exp/size:{n_sim}_fam:{fam_name}_corr:{should_correlate}.pickle', 'wb') as file:
                pickle.dump(res,file, protocol=pickle.HIGHEST_PROTOCOL)
        
        iterator.close()

In [ ]:
############################# Simulation 3: General #############################
sim_fams = [MultiGauss(3,[Identity() for _ in range(3)]),
            MULNOMLSS(4),
            ScaledT(),
            PropHaz([0],[0])]
fam_names = ["MGauss", "Multinomial", "ScaledT", "PropHaz"]

for should_correlate in [True,False]:

    for fam_name, sim_fam in zip(fam_names,sim_fams):
        
        # Set up storage for current sim
        eta_mses = np.zeros((n_sim,1 + len(fcoef_ratios)))
        coverage = np.zeros((n_sim,1 + len(fcoef_ratios)))
        Failures = np.zeros((n_sim,1 + len(fcoef_ratios)))

        # Assign correct family for qEFS estimator
        if isinstance(sim_fam,ScaledT):
            gsmm_fam = GAMLSSGSMMFamily(1,sim_fam)
        elif isinstance(sim_fam,MULNOMLSS):
            gsmm_fam = GAMLSSGSMMFamily(4,sim_fam)
        else:
            gsmm_fam = sim_fam
        
        iterator = tqdm(range(n_sim),desc="Simulating",leave=True)
        for sim_i in iterator:

            # Get data for different families
            if isinstance(sim_fam,ScaledT):
                sim_dat = sim11(5000,2,c=0,seed=sim_i,family=sim_fam,
                                binom_offset = 0,
                                correlate=should_correlate,
                                n_ranef=n_ranef)
                
                # We need to model only the mean: \mu_i
                sim_formula_m = Formula(lhs("y"),
                                    [i(),f(["x0"]),f(["x1"]),f(["x2"]),f(["x3"]),fs(["x0"],rf="x4")],
                                    data=sim_dat)
                
                sim_formulas = [sim_formula_m]

                fcoef = 39 # 37 + 2 for theta

            elif isinstance(sim_fam,PropHaz):
                sim_dat = sim11(5000,2,c=0,seed=sim_i,family=sim_fam,
                                binom_offset = 0.1,
                                correlate=should_correlate,
                                n_ranef=n_ranef)
                sim_dat = sim_dat.sort_values(['y'],ascending=[False])
                sim_dat = sim_dat.reset_index(drop=True)

                u,inv = np.unique(sim_dat["y"],return_inverse=True)
                ut = np.flip(u)
                r = np.abs(inv - max(inv))
                sim_fam = PropHaz(ut=ut,r=r)
                gsmm_fam = copy.deepcopy(sim_fam)

                # Cannot have intercept!
                sim_formula_m = Formula(lhs("delta"),
                                    [f(["x0"]),f(["x1"]),f(["x2"]),f(["x3"]),fs(["x0"],rf="x4")],
                                    data=sim_dat)
                
                sim_formulas = [sim_formula_m]

                fcoef = 36 # 37 - 1 for intercept
            
            elif isinstance(sim_fam,MULNOMLSS):
                # We need to specify K-1 formulas - see the `MULNOMLSS` docstring for details.
                sim_dat = sim18(5000,2,c=0,seed=sim_i,
                                correlate=should_correlate,
                                n_ranef=n_ranef)

                sim_formulas = []
                for k in range(4):
                    terms = [i(),f([f"x{k}"]),fs(["x0"],rf="x4")] if k == 0 else [i(),f([f"x{k}"])]
                    sim_formulas.append(Formula(lhs("y"),terms, data=sim_dat))
                
                fcoef = 40
            else:
                # Multi Gauss case
                sim_dat = sim21(5000,c=0,seed=sim_i,
                                correlate=should_correlate,
                                n_ranef=n_ranef)

                # We need formulas for each mean
                sim_formulas = [
                    Formula(lhs("y0"), [i(), f(["x0"]), fs(["x0"],rf="x4")], data=sim_dat),
                    Formula(lhs("y1"), [i(), f(["x1"]), f(["x2"])], data=sim_dat),
                    Formula(lhs("y2"), [i(), f(["x3"])], data=sim_dat),
                ]

                fcoef = 45 # For means and thetas
            
            sim_dat.to_csv((f"./results/data/sim3_gen/sim_size:{n_sim}_fam:"
                            f"{fam_name}_corr:{should_correlate}_set:{sim_i}.csv"),index=False)

            
            sim_i_failed = [False, *[False for _ in fcoef_ratios]]

            ############################# Fit model with EFS #############################
            if isinstance(sim_fam,ScaledT):
                model_efs = GAMM(sim_formulas[0],sim_fam)
                model_kwargs = copy.deepcopy(default_gamm_kwargs)
            elif isinstance(sim_fam,MULNOMLSS):
                model_efs = GAMMLSS(sim_formulas,sim_fam)
                model_kwargs = copy.deepcopy(default_gammlss_kwargs)
            else:
                model_efs = GSMM(sim_formulas,sim_fam)
                model_kwargs = copy.deepcopy(default_gsmm_kwargs)

            try:
                model_efs.fit(**model_kwargs)
            except:
                sim_i_failed[0] = True

            ############################# Fit model with qEFS #############################
            models = [model_efs]
            for rai,ratio in enumerate(fcoef_ratios):

                qefs_kwargs["structured_qefs_budget"] = int(ratio*fcoef)

                gsmm_model = GSMM(formulas=sim_formulas,family=gsmm_fam)
                
                try:
                    with warnings.catch_warnings():
                        warnings.simplefilter("ignore")
                        gsmm_model.fit(**qefs_kwargs)
                except:
                    sim_i_failed[1+rai] = True

                models.append(gsmm_model)
            
            ######################################## Collect MSEs ####################################

            for mi, model in enumerate(models):

                if sim_i_failed[mi]:
                    print(f"Model {mi+1} failed at {sim_i}")
                    Failures[sim_i,mi] = 1
                    eta_mses[sim_i,mi] = np.nan
                    continue

                # Not converged but not failed outright
                if model.info.code > 0:
                    Failures[sim_i,mi] = 1
                
                # plot(model)

                if len(model.preds) == 1:
                    pred_diff = model.preds[0].flatten() - sim_dat["eta"].values
                    _,mcov,_ = eval_coverage(model,sim_dat,target=sim_dat["eta"].values)
                else:
                    pred_diff = []
                    mcov = []
                    for pi in range(len(model.preds)):
                        pred_diff.extend(model.preds[pi].flatten() - sim_dat[f"eta{pi}"].values)
                        _,_,IN_CI = eval_coverage(model,sim_dat,
                                                dist_par=pi,
                                                target=sim_dat[f"eta{pi}"].values)
                        mcov.extend(IN_CI)
                    mcov = np.sum(mcov) / len(mcov)
                    pred_diff = np.array(pred_diff)
                    
                eta_mses[sim_i,mi] = np.dot(pred_diff,pred_diff)/len(pred_diff)
                coverage[sim_i,mi] = mcov
            
            iterator.set_description_str(desc=f"MSE.: {[(float(np.round(m,decimals=4)),
                                                         float(np.round(sd,decimals=2)))
                                                         for m,sd in zip(np.mean(eta_mses[:(sim_i+1),:],axis=0),
                                                                         np.std(eta_mses[:(sim_i+1),:],axis=0))]}",
                                                                         refresh=True)
            ###################################### Save in progress results ######################################
            res = {"eta_mses":eta_mses,
                   "Failures":Failures,
                   "coverage":coverage,
                   }
            
            with open(f'./results/sim/sim3_gen/size:{n_sim}_fam:{fam_name}_corr:{should_correlate}.pickle', 'wb') as file:
                pickle.dump(res,file, protocol=pickle.HIGHEST_PROTOCOL)
        
        iterator.close()